In [3]:
import os
for f in os.listdir("/content/drive/MyDrive"):
    print(f)

Colab Notebooks
Adsız doküman.gdoc
Başlıksız e-tablo.gsheet
2024 -2025 Format-Motivation-letter-TSHD_ Ilter Calikusu.gdoc
Application for review.gdoc
python notes.gdoc
annem
dosga.gdoc
dosga.docx
Article notes (AI in business) .gdoc
cover letter deloitte.gdoc
Ilter Calikusu CV (1).pdf
motivation letter Ilter Calikusu.gdoc
Cover Letter Ilter Calikusu.gdoc
modernbert_lora
modernbert_final
modernbert_lora_full
modernbert_lora_merged
modernbert_shap_top_200.csv
modernbert_lime_top_200.csv
modernbert_overlap_summary.csv
modernbert_combined_overlap.csv
modernbert_ablation_results.csv
modernbert_final_v2
modernbert_lora_v3


In [4]:
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

base = AutoModelForSequenceClassification.from_pretrained(
    "answerdotai/ModernBERT-base",
    num_labels=2,
    id2label={0:"left", 1:"right"},
    label2id={"left":0, "right":1}
)

model_orig = PeftModel.from_pretrained(
    base,
    "/content/drive/MyDrive/modernbert_lora/checkpoint-1377"
)

merged_orig = model_orig.merge_and_unload()
merged_orig.save_pretrained("/content/drive/MyDrive/modernbert_original_merged")
tokenizer.save_pretrained("/content/drive/MyDrive/modernbert_original_merged")

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/modernbert_original_merged/tokenizer_config.json',
 '/content/drive/MyDrive/modernbert_original_merged/tokenizer.json')

In [8]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

def predict_orig(texts, batch_size=4):
    preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, truncation=True, max_length=1024, padding=True, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = merged_orig(**inputs)
        pred = outputs.logits.argmax(dim=-1).cpu().numpy()
        preds.extend(pred)
    return preds

y_true = test_df["political_leaning"].tolist()
y_pred = [id2label[p] for p in predict_orig(test_df["post_cleaned"].tolist())]

print(classification_report(y_true, y_pred, labels=["left", "right"], zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=["left", "right"]))

train_true = train_df["political_leaning"].tolist()
train_pred = [id2label[p] for p in predict_orig(train_df["post_cleaned"].tolist())]
train_f1 = f1_score(train_true, train_pred, average="macro")
test_f1 = f1_score(y_true, y_pred, average="macro")

              precision    recall  f1-score   support

        left       0.66      0.59      0.62      2555
       right       0.67      0.73      0.70      2869

    accuracy                           0.67      5424
   macro avg       0.66      0.66      0.66      5424
weighted avg       0.67      0.67      0.66      5424

Confusion Matrix:
[[1502 1053]
 [ 763 2106]]
